# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule

A content item should be reviewed if it is old, receives a reasonable amount of traffic, and has a low CTR. Older content with visibility but poor engagement is likely to benefit from refreshing.

### Reason Codes

- STALE_LOWCTR
- STALE_VISIBLE
- LOWCTR_VISIBLE
- REVIEW_PRIORITY

In [6]:
!git clone https://github.com/JaswanthhKumar22/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 113, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 113 (delta 31), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (113/113), 1.85 MiB | 10.18 MiB/s, done.
Resolving deltas: 100% (31/31), done.


In [7]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship/flyrank-ml-internship


In [9]:
import os
print(os.getcwd())

/content/flyrank-ml-internship/flyrank-ml-internship


In [11]:
df.columns

Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',
       'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier',
       'position_tier', 'trend_direction', 'trend_pct'],
      dtype='object')

In [13]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Rule signals
stale = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 500
low_ctr = df["ctr"] < 1

# Baseline score
df["baseline_score"] = (
    stale.astype(int) * 40 +
    visible.astype(int) * 30 +
    low_ctr.astype(int) * 30
)

# Reason codes
conditions = [
    stale & low_ctr,
    stale & visible,
    low_ctr & visible
]

choices = [
    "STALE_LOWCTR",
    "STALE_VISIBLE",
    "LOWCTR_VISIBLE"
]

df["reason_code"] = np.select(
    conditions,
    choices,
    default="REVIEW_PRIORITY"
)

# Action label
df["action"] = "Review Content"

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,5.88,4.55,0.0,good,striking,down,-41.4,60,LOWCTR_VISIBLE,Review Content
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.00,10.00,0.0,good,page_3_5,down,-57.7,60,LOWCTR_VISIBLE,Review Content
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.00,28.57,0.0,good,page_3_5,down,-60.9,60,LOWCTR_VISIBLE,Review Content
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,1.28,3.45,0.0,good,page_1,stable,-13.8,60,LOWCTR_VISIBLE,Review Content
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.00,24.29,0.0,good,page_3_5,down,-34.7,60,LOWCTR_VISIBLE,Review Content


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [16]:
import os

os.makedirs("work/outputs", exist_ok=True)

In [17]:
# Rank queue
df = df.sort_values(
    by="baseline_score",
    ascending=False
).reset_index(drop=True)

# Save CSV
df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully!")

df.head(10)

CSV saved successfully!


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action
0,content_074ba6ead17b,client_d029fa3a95,0.0,0.0,LOW,0.0,keyword article,informational,3994.0,27901.0,...,0.00,50.00,0.0,moderate,page_3_5,down,-36.5,100,STALE_LOWCTR,Review Content
1,content_5feee3994adb,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,transactional,3590.0,22780.0,...,0.00,40.00,0.0,good,page_3_5,down,-89.1,100,STALE_LOWCTR,Review Content
2,content_7f116ae1f6f5,client_9400f1b21c,NaN,NaN,NaN,NaN,keyword article,NaN,1335.0,9375.0,...,0.00,0.00,0.0,moderate,page_1,down,-44.5,100,STALE_LOWCTR,Review Content
3,content_fe16a55cd13d,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3388.0,21742.0,...,2.38,38.64,0.0,good,striking,down,-52.2,100,STALE_LOWCTR,Review Content
4,content_1bfaa38ff26c,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3861.0,24672.0,...,3.75,43.33,0.0,good,page_3_5,down,-74.7,100,STALE_LOWCTR,Review Content
5,content_77d4d5930e5e,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4020.0,26513.0,...,50.00,20.00,0.0,moderate,striking,down,-55.0,100,STALE_LOWCTR,Review Content
6,content_0a91db491d14,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3478.0,21948.0,...,5.13,41.76,0.0,good,striking,down,-51.8,100,STALE_LOWCTR,Review Content
7,content_928af3e22c80,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3118.0,20396.0,...,0.00,0.00,0.0,moderate,striking,down,-45.7,100,STALE_LOWCTR,Review Content
8,content_e3ff1b093148,client_d029fa3a95,0.0,0.0,LOW,0.0,keyword article,informational,4758.0,33575.0,...,0.00,20.00,0.0,moderate,page_1,down,-68.5,100,STALE_LOWCTR,Review Content
9,content_bdbec75c1148,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3696.0,24643.0,...,0.00,33.33,0.0,moderate,page_3_5,stable,-11.7,100,STALE_LOWCTR,Review Content


In [19]:
import os

print(os.path.exists("work/outputs/baseline_action_score.csv"))

True


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [15]:
top20 = df.head(20)

for i, row in top20.iterrows():
    print(f"Rank {i+1}")
    print("Action:", row["action"])
    print("Reason Code:", row["reason_code"])
    print("Confidence: Medium")
    print("What would make it wrong: The recommendation could be incorrect if the content is seasonal, recently updated but not yet reflected in the data, or affected by missing engagement information.")
    print("-" * 70)

Rank 1
Action: Review Content
Reason Code: STALE_LOWCTR
Confidence: Medium
What would make it wrong: The recommendation could be incorrect if the content is seasonal, recently updated but not yet reflected in the data, or affected by missing engagement information.
----------------------------------------------------------------------
Rank 2
Action: Review Content
Reason Code: STALE_LOWCTR
Confidence: Medium
What would make it wrong: The recommendation could be incorrect if the content is seasonal, recently updated but not yet reflected in the data, or affected by missing engagement information.
----------------------------------------------------------------------
Rank 3
Action: Review Content
Reason Code: STALE_LOWCTR
Confidence: Medium
What would make it wrong: The recommendation could be incorrect if the content is seasonal, recently updated but not yet reflected in the data, or affected by missing engagement information.
------------------------------------------------------------

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some high-ranked items may not actually require refreshing.

Possible reasons:

- Seasonal traffic changes
- Temporary CTR fluctuations
- Recently updated content not yet reflected
- Missing values affecting the score

## Leakage Check

No future information was used.

The rule does NOT use:

- trend_pct
- trend_direction
- is_declining_label

These fields are derived from the target and would cause data leakage.

## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.